# Del microdato al indicador: preparación de los microdatos de la GEIH

Este cuaderno lee los archivos que se descargan del DANE y genera los tres archivos CSV que se cargan en la base de datos (persona, ocupado y desocupado). No hay nada que editar a mano: se corre completo y procesa automáticamente todos los periodos que encuentre descargados y todavía no estén en `Datos para BD`.

**Estructura de carpetas**

```
Proyecto GEIH/
├── Microdatos GEIH/        archivos descargados del DANE
│   └── Año/
│       ├── Mes/
└── Datos para BD/          archivos listos para cargar en MySQL
    └── Año/
        ├── Mes/
```

Cada mes se descarga en su carpeta de `Microdatos GEIH` y el resultado queda en la carpeta del mismo año y mes de `Datos para BD`.

## 1. Librerías y acceso a las carpetas

In [1]:
import os
import pandas as pd

# En Colab se conecta Google Drive; en otro entorno esta celda no hace nada
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

## 2. Periodos disponibles

Revisa qué periodos ya tienen los 3 archivos del DANE descargados en `Microdatos GEIH`. Nada que editar.

In [ ]:
RAIZ = r'C:\Users\jessi\OneDrive\Documents\GitHub\Proyecto-ingeniera-de-datos-'    # carpeta que contiene las dos carpetas del proyecto (compartida con Entregable 2)

NOMBRES_MES = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio',
               'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
ARCHIVOS_DANE = [
    'Características generales, seguridad social en salud y educación.CSV',
    'Ocupados.CSV',
    'No ocupados.CSV',
]

def periodos_disponibles():
    """Busca en 'Microdatos GEIH' los periodos que ya tienen los 3 archivos del DANE descargados."""
    base = os.path.join(RAIZ, 'Microdatos GEIH')
    periodos = []
    for anio_str in sorted(os.listdir(base)):
        ruta_anio = os.path.join(base, anio_str)
        if not anio_str.isdigit() or not os.path.isdir(ruta_anio):
            continue
        for mes_idx, nombre_mes in enumerate(NOMBRES_MES, start=1):
            ruta_mes = os.path.join(ruta_anio, nombre_mes)
            if os.path.isdir(ruta_mes) and all(os.path.exists(os.path.join(ruta_mes, a)) for a in ARCHIVOS_DANE):
                periodos.append((int(anio_str), mes_idx))
    return sorted(periodos)

disponibles = periodos_disponibles()
if not disponibles:
    raise SystemExit('No se encontró ningún periodo con los 3 archivos del DANE en "Microdatos GEIH". Descarga al menos un mes antes de correr este notebook.')
print('Periodos encontrados en Microdatos GEIH:', ['%d%02d' % p for p in disponibles])

## 3. Catálogos de apoyo

- **Ciudades:** códigos de `AREA` que tienen territorio propio en la base.
- **Ramas de actividad:** las 13 ramas se arman agrupando las divisiones de dos dígitos de `RAMA2D_R4`. Lo que no esté en la agrupación (vacío, punto o "no sabe, no informa") queda sin rama.

In [ ]:
CIUDADES = [11, 5, 76, 8, 68, 17, 66, 54, 52, 73, 23, 13, 50]

GRUPOS_RAMA = {
    1:  [1, 2, 3],                                             # Agricultura, ganadería, caza, silvicultura y pesca
    2:  [5, 6, 7, 8, 9, 35, 36, 37, 38, 39],                   # Electricidad, gas, agua y gestión de desechos
    3:  list(range(10, 34)),                                   # Industrias manufactureras
    4:  [41, 42, 43],                                          # Construcción
    5:  [45, 46, 47],                                          # Comercio y reparación de vehículos
    6:  [55, 56],                                              # Alojamiento y servicios de comida
    7:  [49, 50, 51, 52, 53],                                  # Transporte y almacenamiento
    8:  [58, 59, 60, 61, 62, 63],                              # Información y comunicaciones
    9:  [64, 65, 66],                                          # Actividades financieras y de seguros
    10: [68],                                                  # Actividades inmobiliarias
    11: [69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 80, 81, 82],  # Profesionales, científicas, técnicas y administrativas
    12: [84, 85, 86, 87, 88],                                  # Administración pública, educación y salud
    13: [90, 91, 92, 93, 94, 95, 96, 97, 98],                  # Artísticas, entretenimiento y otros servicios
}
# Diccionario división -> rama
RAMA = {division: rama for rama, divisiones in GRUPOS_RAMA.items() for division in divisiones}

## 4. Funciones de apoyo

In [ ]:
def a_numero(columna):
    """Convierte texto a número. Lo que no se pueda convertir (vacío, '.') queda como dato faltante."""
    return pd.to_numeric(columna.str.replace(',', '.'), errors='coerce')


def armar_id(tabla):
    """id_persona = directorio (7 dígitos) + secuencia_p (2) + orden (2), con ceros a la izquierda."""
    directorio = a_numero(tabla['DIRECTORIO']).astype('Int64').astype(str).str.zfill(7)
    secuencia = a_numero(tabla['SECUENCIA_P']).astype('Int64').astype(str).str.zfill(2)
    orden = a_numero(tabla['ORDEN']).astype('Int64').astype(str).str.zfill(2)
    return directorio + secuencia + orden

## 5. Procesar y cargar automáticamente

Procesa todos los periodos de `disponibles` y escribe sus CSV en `Datos para BD`, **saltándose los que ya estén procesados**.

In [ ]:
def procesar_periodo(anio, mes):
    """Procesa un periodo completo de la GEIH y escribe los CSV en 'Datos para BD'.
    Si el periodo ya está procesado, no hace nada."""
    id_periodo = anio * 100 + mes
    carpeta_entrada = os.path.join(RAIZ, 'Microdatos GEIH', str(anio), NOMBRES_MES[mes - 1])
    carpeta_salida = os.path.join(RAIZ, 'Datos para BD', str(anio), NOMBRES_MES[mes - 1])
    if os.path.exists(os.path.join(carpeta_salida, 'persona_%d.csv' % id_periodo)):
        print('%d ya está procesado, se omite' % id_periodo)
        return
    os.makedirs(carpeta_salida, exist_ok=True)

    def leer_local(nombre_archivo):
        ruta = os.path.join(carpeta_entrada, nombre_archivo)
        try:
            t = pd.read_csv(ruta, sep=';', decimal=',', encoding='utf-8', dtype=str)
        except UnicodeDecodeError:
            t = pd.read_csv(ruta, sep=';', decimal=',', encoding='latin-1', dtype=str)
        t.columns = [c.strip().upper() for c in t.columns]
        return t

    personas = leer_local('Características generales, seguridad social en salud y educación.CSV')
    ocupados = leer_local('Ocupados.CSV')
    no_ocupados = leer_local('No ocupados.CSV')

    area_cruda = a_numero(personas['AREA'])
    clase_persona = a_numero(personas['CLASE'])
    area_estandarizada = area_cruda.where(area_cruda.isin(CIUDADES), clase_persona)

    persona = pd.DataFrame({
        'idPersona': armar_id(personas),
        'idPeriodo': id_periodo,
        'idSexo': a_numero(personas['P3271']),
        'edad': a_numero(personas['P6040']),
        'clase': clase_persona,
        'area': area_estandarizada,
        'factorExpansion': a_numero(personas['FEX_C18']),
    })
    ocupado = pd.DataFrame({
        'idPersona': armar_id(ocupados),
        'idRama': a_numero(ocupados['RAMA2D_R4']).map(RAMA),
        'horasTrabajo': a_numero(ocupados['P6800']),
        'ingreso': a_numero(ocupados['INGLABO']),
        'cotizaPension': a_numero(ocupados['P6920']).isin([1, 3]).astype(int),
    })
    desocupado = pd.DataFrame({'idPersona': armar_id(no_ocupados[a_numero(no_ocupados['DSI']) == 1])})

    rechazados = []

    def quitar(tabla, nombre, malos, motivo):
        malas = tabla[malos].copy()
        malas['tabla'], malas['motivo'] = nombre, motivo
        rechazados.append(malas)
        return tabla[~malos]

    persona = quitar(persona, 'persona', persona['factorExpansion'].isna() | persona['idPersona'].str.contains('NA'), 'identificador o factor de expansión vacío')
    persona = quitar(persona, 'persona', persona.duplicated('idPersona'), 'identificador repetido')
    persona = quitar(persona, 'persona', ~persona['idSexo'].isin([1, 2]), 'sexo distinto de 1 o 2')
    persona = quitar(persona, 'persona', ~persona['edad'].between(0, 110), 'edad fuera de 0 a 110')
    persona = quitar(persona, 'persona', ~persona['clase'].isin([1, 2]), 'clase distinta de 1 o 2')
    persona = quitar(persona, 'persona', persona['factorExpansion'] <= 0, 'factor de expansión no positivo')

    mayores_15 = persona.loc[persona['edad'] >= 15, 'idPersona']
    ocupado = quitar(ocupado, 'ocupado', ocupado.duplicated('idPersona'), 'identificador repetido')
    ocupado = quitar(ocupado, 'ocupado', ~ocupado['idPersona'].isin(mayores_15), 'no está en persona o es menor de 15 años')
    ocupado = quitar(ocupado, 'ocupado', ocupado['horasTrabajo'] > 168, 'más de 168 horas semanales')
    ocupado = quitar(ocupado, 'ocupado', ocupado['ingreso'] < 0, 'ingreso negativo')

    desocupado = quitar(desocupado, 'desocupado', desocupado.duplicated('idPersona'), 'identificador repetido')
    desocupado = quitar(desocupado, 'desocupado', ~desocupado['idPersona'].isin(mayores_15), 'no está en persona o es menor de 15 años')
    desocupado = quitar(desocupado, 'desocupado', desocupado['idPersona'].isin(ocupado['idPersona']), 'también figura como ocupado')

    for c in ['idPeriodo', 'idSexo', 'edad', 'clase', 'area']:
        persona[c] = persona[c].astype('Int64')
    ocupado['idRama'] = ocupado['idRama'].astype('Int64')

    for nombre, tabla in [('persona', persona), ('ocupado', ocupado), ('desocupado', desocupado)]:
        tabla.to_csv(os.path.join(carpeta_salida, '%s_%d.csv' % (nombre, id_periodo)), index=False, na_rep='', lineterminator='\n')

    todos_rechazados = pd.concat(rechazados)
    if len(todos_rechazados) > 0:
        todos_rechazados.to_csv(os.path.join(carpeta_salida, 'rechazados_%d.csv' % id_periodo), index=False)

    print('%d listo: %d personas, %d ocupados, %d desocupados' % (id_periodo, len(persona), len(ocupado), len(desocupado)))

In [ ]:
for anio, mes in disponibles:
    procesar_periodo(anio, mes)